<a href="https://colab.research.google.com/github/SNayak-166/Evaluation_of_RecommenderSystem/blob/genreencoding/Copy_of_Evaluation_of_RecommenderSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell-1: Loading data from Kaggle. ratings.csv data is of >600mb. GitHub do not allow >25mb files so directly downloading from Kaggle.

Requirement to download: Kaggle username and API key

In [ ]:
import os
import getpass
import shutil
import pandas as pd

destination_path = "ratings.csv"

# Check if ratings.csv is already available in the local Colab workspace
if os.path.exists(destination_path):
    print("--- ratings.csv already exists locally in workspace. Skipping download. ---")
    ratings_df = pd.read_csv(destination_path)
    print(f"File location: {os.path.abspath(destination_path)}")
else:
    print("--- ratings.csv not found locally. Downloading from Kaggle... ---")

    # 1. Securely ask the evaluator for their Kaggle credentials
    if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
        print("--- Kaggle Credentials Required ---")
        os.environ["KAGGLE_USERNAME"] = input("Enter your Kaggle Username: ")
        os.environ["KAGGLE_KEY"] = getpass.getpass("Enter your Kaggle API Key (input will be hidden): ")

    # 2. Install and import kagglehub
    !pip install kagglehub --quiet
    import kagglehub

    # 3. Download the dataset
    path = kagglehub.dataset_download("parasharmanas/movie-recommendation-system")
    print("Dataset downloaded to:", path)

    # 4. Copy ratings.csv to the local Colab workspace for easy access
    source_path = os.path.join(path, "ratings.csv")
    shutil.copy(source_path, destination_path)

    # 5. Load ratings_df from the workspace file
    ratings_df = pd.read_csv(destination_path)
    print("ratings.csv successfully downloaded, saved locally, and loaded!")
    print(f"File location: {os.path.abspath(destination_path)}")

--- ratings.csv already exists locally in workspace. Skipping download. ---
File location: /content/ratings.csv


Cell-3: Storing the processed_movies in the disk.

In [ ]:
import pandas as pd
import os

# Replace with your actual GitHub raw file URL
github_url = "https://raw.githubusercontent.com/SNayak-166/Evaluation_of_RecommenderSystem/refs/heads/genreencoding/processed_movies.csv"

# Load the processed movies dataframe directly from the URL
processed_movies_df = pd.read_csv(github_url)

# Save a local copy of processed_movies.csv in the Google Colab VM workspace
local_path = "processed_movies.csv"
processed_movies_df.to_csv(local_path, index=False)

print(f"processed_movies.csv successfully saved to Colab VM!")
print(f"File location: {os.path.abspath(local_path)}")

processed_movies.csv successfully saved to Colab VM!
File location: /content/processed_movies.csv


Cell-4: This cell splits the consolidated interaction dataset (`ratings_df`) into a synchronized 67% training set and a 33% testing set using a fixed random seed. Both splits are stored directly on the local VM disk to conserve system RAM, enabling independent downstream models to draw from identical training profiles.

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import gc

print("--- Splitting ratings_df into synchronized train/test (67/33) sets... ---")
train_df, test_df = train_test_split(ratings_df, test_size=0.33, random_state=42)

print("Saving splits to disk as CSV files...")
# Note: We split ratings only to save disk space and prevent redundancy.
# The genre features from processed_movies_df are merged on-the-fly during model training (Step 2).
train_df.to_csv('train_ratings.csv', index=False)
test_df.to_csv('test_ratings.csv', index=False)

print("Splits successfully written to disk!")
print(f"-> Train Set size: {len(train_df):,} samples (saved to 'train_ratings.csv')")
print(f"-> Test Set size: {len(test_df):,} samples (saved to 'test_ratings.csv')")
print("Note: Genre features will be joined dynamically on 'movieId' during training to conserve RAM and storage.")

del train_df, test_df
gc.collect()
print("RAM cleared of split variables!")

--- Splitting ratings_df into synchronized train/test (67/33) sets... ---
Saving splits to disk as CSV files...


KeyboardInterrupt: 

Cell-5: This cell incrementally trains both **Naive Bayes** and **SGD Logistic Regression** models. It reads the shared `train_ratings.csv` in memory-efficient chunks, maps movie genres, fits the models, and persists them directly to disk as `.pkl` files.

In [ ]:
import gc
import joblib
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
import pandas as pd
import numpy as np

# Define the static movie genre feature columns directly inside the training cell
genre_columns = [
    '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children',
    'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
    'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
    'Thriller', 'War', 'Western'
]

lr_sgd = SGDClassifier(loss='log_loss', penalty='l2', alpha=0.0001, random_state=42)
nb_model = MultinomialNB()

chunk_size = 150000
print("Starting multi-batch training on Content-Based models using genre features...")

for chunk_idx, chunk in enumerate(pd.read_csv('train_ratings.csv', chunksize=chunk_size)):
    # Join interactions chunk with the local processed_movies metadata dynamically on movieId
    chunk_merged = pd.merge(chunk, processed_movies_df, on='movieId')
    chunk_merged['liked'] = (chunk_merged['rating'] >= 4.0).astype(int)

    X_chunk = chunk_merged[genre_columns]
    y_chunk = chunk_merged['liked']

    if chunk_idx == 0:
        nb_model.partial_fit(X_chunk, y_chunk, classes=np.array([0, 1]))
    else:
        nb_model.partial_fit(X_chunk, y_chunk)

    lr_sgd.partial_fit(X_chunk, y_chunk, classes=np.array([0, 1]))

    del chunk, chunk_merged, X_chunk, y_chunk
    gc.collect()

print("Saving Content-Based models to disk...")
joblib.dump(nb_model, 'nb_model.pkl')
joblib.dump(lr_sgd, 'lr_sgd.pkl')

del nb_model, lr_sgd
gc.collect()
print("Content-Based models saved and RAM cleared!")

Cell-6: This cell reads our standardized training CSV data file into the `surprise` dataset framework and trains an SVD (Singular Value Decomposition) Collaborative Filtering model. The model is saved to disk immediately to free up active memory.

In [ ]:
!pip install scikit-surprise

In [ ]:
import sys
import subprocess

try:
    from surprise import Dataset, Reader, SVD
except ImportError:
    print("--- surprise library not found in active path. Installing scikit-surprise dynamically... ---")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-surprise"])
    from surprise import Dataset, Reader, SVD

import joblib
import os
import gc

reader = Reader(line_format='user item rating timestamp', sep=',', rating_scale=(0.5, 5.0), skip_lines=1)
train_csv_path = os.path.join('/content', 'train_ratings.csv')

print("Loading train_ratings.csv into SVD trainset...")
data = Dataset.load_from_file(train_csv_path, reader=reader)
trainset = data.build_full_trainset()

print("Training SVD model...")
svd_model = SVD(n_factors=30, n_epochs=15, lr_all=0.005, reg_all=0.02, random_state=42)
svd_model.fit(trainset)

print("Saving SVD model to disk...")
joblib.dump(svd_model, 'svd_model.pkl')

del data, trainset, svd_model
gc.collect()
print("SVD model saved to disk and RAM cleared!")

Cell-7: This cell trains a Sparse Linear Methods (SLIM) model over the standardized training set. To protect system RAM limits, it solves regression models for a subset of the target items, writes the learned similarity weight matrix and ID dictionaries to CSVs, and frees all memory structures.

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.linear_model import ElasticNet
import os
import gc

print("Loading train_ratings.csv for SLIM...")
train_ratings = pd.read_csv('train_ratings.csv')

n_users = train_ratings['userId'].nunique()
n_movies = train_ratings['movieId'].nunique()

users = train_ratings['userId'].astype('category')
movies = train_ratings['movieId'].astype('category')

user_item_matrix = csr_matrix(
    (train_ratings['rating'], (users.cat.codes, movies.cat.codes)),
    shape=(n_users, n_movies)
).tocsc()

num_items = user_item_matrix.shape[1]
max_items_to_train = min(num_items, 500)
W = np.zeros((max_items_to_train, num_items))

print(f"Training SLIM model on {max_items_to_train} target items...")
for j in range(max_items_to_train):
    y_col = user_item_matrix[:, j].toarray().ravel()
    left_cols = user_item_matrix[:, :j]
    right_cols = user_item_matrix[:, j+1:]
    X_cols = hstack([left_cols, right_cols]).tocsr()

    en = ElasticNet(alpha=0.1, l1_ratio=0.5, positive=True, fit_intercept=False, max_iter=300)
    en.fit(X_cols, y_col)

    coefs = en.coef_
    W[j, :j] = coefs[:j]
    W[j, j+1:] = coefs[j:]

print("Saving SLIM weights and mappings to disk...")
pd.DataFrame(W).to_csv("slim_weights.csv", index=False, header=False)

user_map = {uid: code for code, uid in enumerate(users.cat.categories)}
movie_map = {mid: code for code, mid in enumerate(movies.cat.categories)}
pd.DataFrame(list(user_map.items()), columns=['userId', 'code']).to_csv("slim_user_map.csv", index=False)
pd.DataFrame(list(movie_map.items()), columns=['movieId', 'code']).to_csv("slim_movie_map.csv", index=False)

del train_ratings, user_item_matrix, W, user_map, movie_map, users, movies
gc.collect()
print("SLIM training finalized and RAM cleared!")

unified evaluation code that evaluates all four models on the exact same footing—outputting RMSE, MAE, Accuracy, Precision, Recall, F1-Score, and ROC-AUC for everyone

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, mean_squared_error, mean_absolute_error
from surprise import Dataset, Reader
import numpy as np
import pandas as pd
import gc

print("=== STARTING COMPLETE UNIFIED EVALUATION PHASE ===")

# 1. Load the test ratings dataset directly from the local workspace
print("Loading test_ratings.csv...")
test_ratings = pd.read_csv('test_ratings.csv')
print(f"Loaded {len(test_ratings):,} test rows successfully.\n")

evaluation_summary = {}

# Prepare common binarized test targets (Threshold >= 4.0)
test_merged = pd.merge(test_ratings, processed_movies_df, on='movieId')
test_merged['liked'] = (test_merged['rating'] >= 4.0).astype(int)

X_test_eval = test_merged[genre_columns]
y_test_eval = test_merged['liked']
y_test_ratings = test_merged['rating'] # True continuous ratings for RMSE/MAE

# -------------------------------------------------------------
# 2. EVALUATE CONTENT-BASED MODELS (Naive Bayes & Logistic Regression)
# -------------------------------------------------------------
print("Evaluating Naive Bayes and Logistic Regression (Classification & Rating Errors)...")

for model_name, clf_model in [("Naive Bayes", nb_model), ("Logistic Regression", lr_sgd)]:
    # Binary predictions
    preds = clf_model.predict(X_test_eval)
    report = classification_report(y_test_eval, preds, output_dict=True, zero_division=0)

    # Get probabilities for ROC-AUC and continuous rating mapping
    try:
        probs = clf_model.predict_proba(X_test_eval)[:, 1]
        auc = roc_auc_score(y_test_eval, probs)

        # Map probabilities [0, 1] to continuous rating scale [0.5, 5.0] for RMSE/MAE
        mapped_ratings = 0.5 + (probs * 4.5)
        cb_rmse = np.sqrt(mean_squared_error(y_test_ratings, mapped_ratings))
        cb_mae = mean_absolute_error(y_test_ratings, mapped_ratings)
    except:
        auc = "N/A"
        cb_rmse = "N/A"
        cb_mae = "N/A"

    evaluation_summary[model_name] = {
        "Accuracy": round(report['accuracy'], 4),
        "Precision (Liked)": round(report['1']['precision'], 4),
        "Recall (Liked)": round(report['1']['recall'], 4),
        "F1-Score (Liked)": round(report['1']['f1-score'], 4),
        "ROC-AUC": round(auc, 4) if isinstance(auc, float) else auc,
        "RMSE": round(cb_rmse, 4) if isinstance(cb_rmse, float) else cb_rmse,
        "MAE": round(cb_mae, 4) if isinstance(cb_mae, float) else cb_mae
    }

# -------------------------------------------------------------
# 3. EVALUATE SVD COLLABORATIVE FILTERING
# -------------------------------------------------------------
print("Evaluating SVD Model...")
reader = Reader(rating_scale=(0.5, 5.0))
test_data_svd = Dataset.load_from_df(test_ratings[['userId', 'movieId', 'rating']], reader)
svd_testset = test_data_svd.build_full_testset()

svd_predictions = svd_model.test(svd_testset)

svd_y_true = [r for (_, _, r, _, _) in svd_predictions]
svd_y_pred = [est for (_, _, _, est, _) in svd_predictions]

svd_rmse = np.sqrt(mean_squared_error(svd_y_true, svd_y_pred))
svd_mae = mean_absolute_error(svd_y_true, svd_y_pred)

svd_true_bin = [1 if r >= 4.0 else 0 for r in svd_y_true]
svd_pred_bin = [1 if est >= 4.0 else 0 for est in svd_y_pred]
svd_cls = classification_report(svd_true_bin, svd_pred_bin, output_dict=True, zero_division=0)

evaluation_summary["SVD"] = {
    "Accuracy": round(svd_cls['accuracy'], 4),
    "Precision (Liked)": round(svd_cls['1']['precision'], 4),
    "Recall (Liked)": round(svd_cls['1']['recall'], 4),
    "F1-Score (Liked)": round(svd_cls['1']['f1-score'], 4),
    "ROC-AUC": "N/A",
    "RMSE": round(svd_rmse, 4),
    "MAE": round(svd_mae, 4)
}

# -------------------------------------------------------------
# 4. EVALUATE SLIM COLLABORATIVE FILTERING
# -------------------------------------------------------------
print("Evaluating SLIM Model...")
slim_y_true = []
slim_y_pred = []

# Reference mapping categories
train_ratings_ref = pd.read_csv('ratings.csv', nrows=100000)
user_map = {uid: code for code, uid in enumerate(train_ratings_ref['userId'].astype('category').cat.categories)}
movie_map = {mid: code for code, mid in enumerate(train_ratings_ref['movieId'].astype('category').cat.categories)}
del train_ratings_ref

for _, row in test_ratings.iterrows():
    u_id = row['userId']
    m_id = row['movieId']
    actual_rating = row['rating']

    if u_id in user_map and m_id in movie_map:
        u_idx = user_map[u_id]
        m_idx = movie_map[m_id]

        if m_idx < max_items_to_train:
            user_profile = user_item_matrix.getrow(u_idx).toarray().ravel()
            predicted_rating = np.dot(user_profile, W[:, m_idx])

            if predicted_rating > 0:
                slim_y_true.append(actual_rating)
                slim_y_pred.append(predicted_rating)

if len(slim_y_true) > 0:
    slim_rmse = np.sqrt(mean_squared_error(slim_y_true, slim_y_pred))
    slim_mae = mean_absolute_error(slim_y_true, slim_y_pred)

    slim_true_bin = [1 if r >= 4.0 else 0 for r in slim_y_true]
    slim_pred_bin = [1 if p >= 4.0 else 0 for p in slim_y_pred]
    slim_cls = classification_report(slim_true_bin, slim_pred_bin, output_dict=True, zero_division=0)

    evaluation_summary["SLIM"] = {
        "Accuracy": round(slim_cls['accuracy'], 4),
        "Precision (Liked)": round(slim_cls['1']['precision'], 4),
        "Recall (Liked)": round(slim_cls['1']['recall'], 4),
        "F1-Score (Liked)": round(slim_cls['1']['f1-score'], 4),
        "ROC-AUC": "N/A",
        "RMSE": round(slim_rmse, 4),
        "MAE": round(slim_mae, 4)
    }
else:
    evaluation_summary["SLIM"] = {"Status": "Insufficient overlap in tested slice"}

# -------------------------------------------------------------
# 5. DISPLAY UNIFIED COMPARATIVE SUMMARY TABLE
# -------------------------------------------------------------
summary_df = pd.DataFrame(evaluation_summary).T
print("\n======================================================================")
print("             COMPLETE UNIFIED MODEL PERFORMANCE SUMMARY               ")
print("======================================================================")
print(summary_df.to_string())
print("======================================================================")

# Clean up memory
del test_ratings, test_merged, svd_predictions
gc.collect()